# 08 — Параллельный CrafText rollout pipeline

Этот пример показывает текущую JAX-native параллельность: `B` независимых CrafText сред сбрасываются и шагают через `jax.vmap`, а `T` шагов каждого rollout собираются одним `jax.lax.scan`. Результат — time-major action/reward tensors `[T, B]`.

> Текущий `QwenTunixBackend` преднамеренно принимает один prompt за вызов. Поэтому здесь **не имитируется** параллельная генерация LLM через threads: это дало бы ложный throughput и нарушило бы границу backend. Следующий production шаг — batched Tunix/RLCluster rollout service с versioned policy lag/back-pressure contract.


In [ ]:
from dataclasses import replace
from pathlib import Path
from time import perf_counter

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from tunix_craftext.config import load_mvp_config
from tunix_craftext.random_policy import sample_masked_actions
from tunix_craftext.runtime import build_craftext_runtime


In [ ]:
ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run from inside the tunix-craftext repository.')

BATCH_SIZE = 8
HORIZON = 32
base = load_mvp_config(ROOT / 'configs' / 'mvp' / 'tiny_craftext.yaml')
config = replace(base, environment=replace(base.environment, batch_size=BATCH_SIZE, horizon=HORIZON))
runtime = build_craftext_runtime(config)
print('JAX backend:', jax.default_backend())
print('devices:', jax.devices())
print('parallel environments:', BATCH_SIZE, 'horizon:', HORIZON)


In [ ]:
def rollout(seed: int) -> tuple[jax.Array, jax.Array]:
    """Collect [T, B] random valid actions and rewards in one compiled execution."""
    reset_keys = jax.random.split(jax.random.PRNGKey(seed), BATCH_SIZE)
    reset = jax.vmap(runtime.adapter.reset)(reset_keys)
    initial_mask = jnp.broadcast_to(reset.action_mask, (BATCH_SIZE, runtime.action_count))
    keys = jax.random.split(
        jax.random.PRNGKey(seed + 1), HORIZON * 2 * BATCH_SIZE
    ).reshape(HORIZON, 2, BATCH_SIZE, 2)

    def one_step(carry, step_keys):
        state, action_mask = carry
        actions = sample_masked_actions(step_keys[0], action_mask)
        transition = jax.vmap(runtime.adapter.step)(step_keys[1], state, actions)
        return (transition.state, transition.action_mask), (actions, transition.reward)

    _, (actions, rewards) = jax.lax.scan(one_step, (reset.state, initial_mask), keys)
    return actions, rewards

compiled_rollout = jax.jit(rollout)
compile_started = perf_counter()
actions, rewards = compiled_rollout(config.run.seed)
rewards.block_until_ready()
compile_and_first_ms = (perf_counter() - compile_started) * 1_000

steady_started = perf_counter()
actions, rewards = compiled_rollout(config.run.seed + 100)
rewards.block_until_ready()
steady_ms = (perf_counter() - steady_started) * 1_000
print(f'compile + first execution: {compile_and_first_ms:.2f} ms')
print(f'steady execution: {steady_ms:.2f} ms')
print(f'env steps/s: {BATCH_SIZE * HORIZON / (steady_ms / 1_000):.1f}')


In [ ]:
figure, (action_axis, reward_axis) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
action_image = action_axis.imshow(actions.T, aspect='auto', interpolation='nearest')
action_axis.set_ylabel('environment index')
action_axis.set_title('Sampled valid actions [environment, time]')
figure.colorbar(action_image, ax=action_axis, label='action id')

reward_image = reward_axis.imshow(rewards.T, aspect='auto', interpolation='nearest', cmap='RdYlGn')
reward_axis.set_xlabel('rollout step')
reward_axis.set_ylabel('environment index')
reward_axis.set_title('Environment rewards [environment, time]')
figure.colorbar(reward_image, ax=reward_axis, label='reward')
figure.tight_layout()
plt.show()


## Как это станет LLM rollout pipeline

1. Каждый env worker формирует typed `PromptContext` из своего `EnvState`; MegaPrompts render сейчас остаётся host-side.
2. Batch prompt tokens идут в batched Tunix actor/rollout role, который возвращает completions, token ids и logprobs с policy version.
3. Strict decode/fallback создаёт `[B]` action ids; этот notebook уже показывает последующий JAX-parallel `vmap(step)` transport.
4. Flashbax принимает fixed-shape trajectory items; learner фиксирует policy lag, queue depth и end-to-end phase timing.

Пока этот bridge не реализован и hardware-gated, для LLM latency используйте `make perf-text`: он даёт честный phase trace однопоточного Qwen/CrafText пути.
